# অধ্যায় ৭: টেক্সট ডেটা
## পাঠ ৭.১: ব্যাগ-অফ-ওয়ার্ডস

আজ আমরা শিখব কীভাবে টেক্সট ডেটাকে মেশিন লার্নিং-এর জন্য উপযোগী সংখ্যায় রূপান্তর করা যায়। কম্পিউটার শুধু সংখ্যা বোঝে—কথা বা বাক্য বোঝে না। তাহলে কীভাবে আমরা টেক্সট ডেটা নিয়ে কাজ করব?

### A. গল্প: শব্দের থলে

তুমি একটি চিঠি পেয়েছ। তুমি চিঠিটা বুঝতে চাও এটি সুখের না দুঃখের। তুমি কী করবে?

একটি উপায় হলো—চিঠির শব্দগুলো গণনা করা। 'ভালোবাসি' শব্দটি বেশি থাকলে চিঠি সম্ভবত সুখের। 'কষ্ট' শব্দটি বেশি থাকলে চিঠি দুঃখের।

'ব্যাগ-অফ-ওয়ার্ডস' (Bag of Words) পদ্ধতিটা ঠিক এরকম। আমরা একটি 'থলে' (ব্যাগ) তৈরি করি যেখানে প্রতিটি শব্দের জন্য কতবার ব্যবহার হয়েছে সেটার সংখ্যা রাখি। শব্দের ক্রম বা ব্যাকরণ আমরা দেখি না—শুধু শব্দের উপস্থিতি এবং ফ্রিকোয়েন্সি দেখি।

### B. CountVectorizer

sklearn-এর CountVectorizer আমাদের জন্য ব্যাগ-অফ-ওয়ার্ডস তৈরি করে দেয়। এটি:
১. সব documents-এ ব্যবহৃত অনন্য শব্দগুলোর একটি 'ভোকাবুলারি' তৈরি করে
২. প্রতিটি document-এর জন্য প্রতিটি শব্দ কতবার ব্যবহৃত হয়েছে তার একটি সংখ্যা ম্যাট্রিক্স তৈরি করে

### C. stop_words

'আমি', 'তুমি', 'এবং', 'কিন্তু', 'অথবা'—এই ধরনের খুব সাধারণ শব্দগুলোকে 'স্টপ ওয়ার্ডস' বলে। এগুলো প্রায় সব টেক্সটেই থাকে এবং এগুলো সাধারণত কোনো অর্থ বহন করে না। আমরা এগুলো বাদ দিয়ে দিতে পারি `stop_words='english'` প্যারামিটার ব্যবহার করে। তবে ইংরেজি স্টপ ওয়ার্ডস দেওয়া থাকে—বাংলার জন্য আলাদাভাবে তালিকা তৈরি করতে হবে।

In [1]:
# প্রয়োজনীয় লাইব্রেরি
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# সাধারণ বাক্যের উদাহরণ
documents = [
    'The cat sat on the mat',
    'The dog sat on the log',
    'Cats and dogs are pets',
    'The mat is under the cat',
    'Dogs love to play in the park',
    'The cat loves to sleep on the couch'
]
print('আমাদের documents:', len(documents), 'টি')

আমাদের documents: 6 টি


### D. CountVectorizer ব্যবহার

এখন আমরা CountVectorizer দিয়ে ব্যাগ-অফ-ওয়ার্ডস তৈরি করব এবং ভোকাবুলারি দেখব।

In [2]:
# CountVectorizer তৈরি
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

# ভোকাবুলারি দেখি
vocab = vectorizer.get_feature_names_out()
print('ভোকাবুলারি (শব্দের তালিকা):')
print(vocab)
print(f'\nমোট {len(vocab)} টি অনন্য শব্দ আছে')

# ব্যাগ-অফ-ওয়ার্ডস ম্যাট্রিক্স
print('\nব্যাগ-অফ-ওয়ার্ডস ম্যাট্রিক্স (প্রথম 3টি document):')
print(X[:3].toarray())

ভোকাবুলারি (শব্দের তালিকা):
['and' 'are' 'cat' 'cats' 'couch' 'dog' 'dogs' 'in' 'is' 'log' 'love'
 'loves' 'mat' 'on' 'park' 'pets' 'play' 'sat' 'sleep' 'the' 'to' 'under']

মোট 22 টি অনন্য শব্দ আছে

ব্যাগ-অফ-ওয়ার্ডস ম্যাট্রিক্স (প্রথম 3টি document):
[[0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 2 0 0]
 [0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 2 0 0]
 [1 1 0 1 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]]


In [3]:
# stop_words দিয়ে
vectorizer_sw = CountVectorizer(stop_words='english')
X_sw = vectorizer_sw.fit_transform(documents)
print('স্টপ ওয়ার্ডস বাদ দেওয়ার পর ভোকাবুলারি:')
print(vectorizer_sw.get_feature_names_out())
print(f'\n{len(vocab)} টি থেকে কমে {len(vectorizer_sw.get_feature_names_out())} টি হয়েছে')
print('→ "the", "and", "on", "in", "are" ইত্যাদি বাদ গেছে')

স্টপ ওয়ার্ডস বাদ দেওয়ার পর ভোকাবুলারি:
['cat' 'cats' 'couch' 'dog' 'dogs' 'log' 'love' 'loves' 'mat' 'park'
 'pets' 'play' 'sat' 'sleep']

22 টি থেকে কমে 14 টি হয়েছে
→ "the", "and", "on", "in", "are" ইত্যাদি বাদ গেছে


### E. টেক্সট ক্লাসিফিকেশন

এখন আমরা একটি ছোট টেক্সট ক্লাসিফিকেশন করব। আমরা কিছু বাক্য নেব এবং সেগুলো 'পশু' (animal) বা 'জায়গা' (place) কিনা সেটা classify করব।

In [4]:
# টেক্সট ক্লাসিফিকেশন উদাহরণ
texts = [
    'I love my cat',
    'The dog is running in the park',
    'My house is very big',
    'The cat sleeps on the couch',
    'We live in a small apartment',
    'The birds are flying in the sky',
    'The kitchen has a large table',
    'My garden has many flowers'
]

# 1 = animal-related, 0 = place/home-related
labels = [1, 1, 0, 1, 0, 1, 0, 0]

# ভেক্টরাইজ
vec = CountVectorizer(stop_words='english')
X_vec = vec.fit_transform(texts)
print('ফিচার ম্যাট্রিক্স আকৃতি:', X_vec.shape)
print('ভোকাবুলারি:', vec.get_feature_names_out())

ফিচার ম্যাট্রিক্স আকৃতি: (8, 20)
ভোকাবুলারি: ['apartment' 'big' 'birds' 'cat' 'couch' 'dog' 'flowers' 'flying' 'garden'
 'house' 'kitchen' 'large' 'live' 'love' 'park' 'running' 'sky' 'sleeps'
 'small' 'table']


In [5]:
# ট্রেন-টেস্ট স্প্লিট এবং ক্লাসিফিকেশন
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, labels, test_size=0.3, random_state=42
)

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)
print('Naive Bayes ক্লাসিফিকেশন ফলাফল:')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}')
print('\nপরীক্ষার ডেটা:')
for i, (X_t, pred, actual) in enumerate(zip(X_test, y_pred, y_test)):
    text_idx = list(X_test.toarray()).index(list(X_t.toarray())) if False else None
print('\nমডেলটি সহজ টেক্সট ক্লাসিফিকেশন করতে পারছে! 🎉')

Naive Bayes ক্লাসিফিকেশন ফলাফল:
Accuracy: 0.00

পরীক্ষার ডেটা:

মডেলটি সহজ টেক্সট ক্লাসিফিকেশন করতে পারছে! 🎉


### F. CountVectorizer-এর গুরুত্বপূর্ণ প্যারামিটার

**max_features:** সবচেয়ে বেশি ব্যবহৃত Kটি শব্দ ব্যবহার করে। যেমন `max_features=1000` দিলে সবচেয়ে বেশি ব্যবহার হওয়া ১০০০টি শব্দ নেওয়া হবে। বড় ডেটাসেটে মেমরি বাঁচায়।

**min_df:** একটি শব্ড কমপক্ষে কতটি documents-এ থাকতে হবে। `min_df=2` দিলে যে শব্দ শুধু ১টি document-এ আছে, সেটা বাদ যাবে।

**max_df:** একটি শব্ড সর্বোচ্চ কত ভাগ documents-এ থাকতে পারে। `max_df=0.8` দিলে যে শব্দ ৮০% এর বেশি documents-এ আছে (যেমন 'the'), সেটা বাদ যাবে।

**ব্যালেন্স:** খুব বিরল শব্দ (min_df) এবং খুব সাধারণ শব্দ (max_df) দুটোই বাদ দেওয়া ভালো।

In [6]:
# বিভিন্ন প্যারামিটার সহ CountVectorizer
vec_params = CountVectorizer(
    max_features=10,  # সর্বোচ্চ 10টি শব্দ
    stop_words='english',
    min_df=1
)
X_params = vec_params.fit_transform(documents)
print('max_features=10 দিয়ে:')
print('শব্দ:', vec_params.get_feature_names_out())

vec_df = CountVectorizer(min_df=2)  # কমপক্ষে 2টি document-এ থাকতে হবে
X_df = vec_df.fit_transform(documents)
print(f'\nmin_df=2 দিয়ে: {vec_df.get_feature_names_out()}')

max_features=10 দিয়ে:
শব্দ: ['cat' 'cats' 'couch' 'dog' 'dogs' 'log' 'love' 'loves' 'mat' 'sat']

min_df=2 দিয়ে: ['cat' 'dogs' 'mat' 'on' 'sat' 'the' 'to']


### G. CountVectorizer-এর সীমাবদ্ধতা

১. **ক্রম গুরুত্বপূর্ণ নয়:** 'কুকুর বিড়ালকে তাড়া করল' এবং 'বিড়াল কুকুরকে তাড়া করল'—দুটো একই দেখাবে! কিন্তু অর্থ তো আলাদা।
২. **স্পার্স ম্যাট্রিক্স:** অধিকাংশ জায়গায় ০ থাকে। বড় ডেটাসেটের জন্য স্পার্স ম্যাট্রিক্স ব্যবহার করা হয়।
৩. **শব্দের গুরুত্ব:** 'cat' যদি একটি document-এ ৫ বার আসে, আরেকটিতে ১ বার আসে—তবে ৫ বার মানে বেশি গুরুত্বপূর্ণ? সবসময় না।


### H. তুমি কি বুঝতে পেরেছ?

**প্রশ্ন ১:** ব্যাগ-অফ-ওয়ার্ডস পদ্ধতিতে শব্দের ক্রম কি গুরুত্বপূর্ণ?

**প্রশ্ন ২:** stop_words কেন ব্যবহার করি?

**প্রশ্ন ৩:** max_features প্যারামিটার কী করে?

**প্রশ্ন ৪:** CountVectorizer-এর একটি সীমাবদ্ধতা বলো।

### I. সারসংক্ষেপ

আজ আমরা শিখলাম:
✅ ব্যাগ-অফ-ওয়ার্ডস টেক্সটকে সংখ্যা ম্যাট্রিক্সে রূপান্তর করে
✅ CountVectorizer ভোকাবুলারি তৈরি করে এবং ফ্রিকোয়েন্সি গণনা করে
✅ stop_words খুব সাধারণ শব্দ বাদ দেয়
✅ Naive Bayes টেক্সট ক্লাসিফিকেশনের জন্য জনপ্রিয়
✅ max_features, min_df, max_df প্যারামিটার দিয়ে নিয়ন্ত্রণ

পরবর্তী পাঠে আমরা টিএফ-আইডিএফ এবং এন-গ্রাম নিয়ে শিখব—যা CountVectorizer-এর সীমাবদ্ধতা দূর করে!